In [ ]:
# FireWatch AI: PPE Detection Model Training

In this notebook, I will prepare the separate YOLO model for personal protective equipment (PPE) detection. The model will help identify people, protective equipment, and the PPE-related classes defined in the prepared dataset.

I will use the verified PPE candidate dataset, preserve the original dataset, and save all training artifacts in a separate run directory. I will verify the dataset configuration before starting any training.

In [1]:
from pathlib import Path
from google.colab import drive

# Mount the existing Google Drive.
drive.mount("/content/drive", force_remount=False)

# Define FireWatch AI project paths.
PROJECT_ROOT = Path("/content/drive/MyDrive/FireWatch-AI")

DATASET_DIR = (
    PROJECT_ROOT
    / "datasets/candidates/ppe_compliance_v2_bbox_candidate"
)

DATA_YAML = DATASET_DIR / "data.yaml"

print("Project exists:", PROJECT_ROOT.is_dir())
print("PPE candidate dataset exists:", DATASET_DIR.is_dir())
print("PPE dataset YAML exists:", DATA_YAML.is_file())

if not all([
    PROJECT_ROOT.is_dir(),
    DATASET_DIR.is_dir(),
    DATA_YAML.is_file(),
]):
    raise FileNotFoundError(
        "PPE dataset or configuration not found. "
        "Check the existing project paths before continuing."
    )

print("\nPPE dataset path check: PASS")


Mounted at /content/drive
Project exists: True
PPE candidate dataset exists: True
PPE dataset YAML exists: True

PPE dataset path check: PASS


In [2]:
from pathlib import Path
import yaml

# Load the PPE dataset configuration.
with open(DATA_YAML, "r") as file:
    ppe_config = yaml.safe_load(file)

print("Dataset YAML:", DATA_YAML)
print("\nNumber of classes:", ppe_config.get("nc"))
print("Class names:", ppe_config.get("names"))

# Check that the YAML points to the expected dataset directory.
yaml_root = Path(ppe_config.get("path", DATASET_DIR))
if not yaml_root.is_absolute():
    yaml_root = DATASET_DIR / yaml_root

print("\nChecking dataset splits:")
split_status = {}

for split in ("train", "val", "test"):
    split_path = ppe_config.get(split)

    if not split_path:
        split_status[split] = False
        print(f"{split}: MISSING from YAML")
        continue

    split_path = Path(split_path)
    if not split_path.is_absolute():
        split_path = yaml_root / split_path

    exists = split_path.is_dir()
    split_status[split] = exists
    print(f"{split}: {split_path} | Exists: {exists}")

# Verify the expected 13-class PPE mapping.
expected_classes = [
    "Boots", "Gloves", "Goggles", "Helmet", "Mask",
    "NO-Boots", "NO-Gloves", "NO-Goggles", "NO-Helmet",
    "NO-Mask", "NO-Vest", "Person", "Vest"
]

actual_names = ppe_config.get("names", [])
if isinstance(actual_names, dict):
    actual_names = [
        actual_names[key]
        for key in sorted(actual_names, key=lambda key: int(key))
    ]

print("\nClass count matches:", len(actual_names) == 13)
print("Class mapping matches:", actual_names == expected_classes)
print("All split paths exist:", all(split_status.values()))

Dataset YAML: /content/drive/MyDrive/FireWatch-AI/datasets/candidates/ppe_compliance_v2_bbox_candidate/data.yaml

Number of classes: 13
Class names: ['Boots', 'Gloves', 'Goggles', 'Helmet', 'Mask', 'NO-Boots', 'NO-Gloves', 'NO-Goggles', 'NO-Helmet', 'NO-Mask', 'NO-Vest', 'Person', 'Vest']

Checking dataset splits:
train: /content/drive/MyDrive/FireWatch-AI/datasets/candidates/ppe_compliance_v2_bbox_candidate/train/images | Exists: True
val: /content/drive/MyDrive/FireWatch-AI/datasets/candidates/ppe_compliance_v2_bbox_candidate/valid/images | Exists: True
test: /content/drive/MyDrive/FireWatch-AI/datasets/candidates/ppe_compliance_v2_bbox_candidate/test/images | Exists: True

Class count matches: True
Class mapping matches: True
All split paths exist: True


In [3]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 21.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [ultralytics]


In [4]:
# I am checking the training environment before preparing the PPE model.
# This cell does not start training or change my dataset.

import torch
import ultralytics

print("Ultralytics version:", ultralytics.__version__)
print("PyTorch version:", torch.__version__)

# A GPU is recommended for training this dataset efficiently.
gpu_available = torch.cuda.is_available()
print("GPU available:", gpu_available)

if gpu_available:
    print("GPU device:", torch.cuda.get_device_name(0))
else:
    print("GPU unavailable: environment checks can continue on CPU.")
    print("I will wait for GPU access before starting PPE training.")

print("\nTraining environment check: COMPLETE")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics version: 8.4.190
PyTorch version: 2.11.0+cpu
GPU available: False
GPU unavailable: environment checks can continue on CPU.
I will wait for GPU access before starting PPE training.

Training environment check: COMPLETE


In [6]:
import json

# I am reducing the training duration to make the first
# PPE experiment faster while keeping the other settings fixed.

PPE_CONFIG["epochs"] = 20
PPE_CONFIG["patience"] = 5

# Save the updated settings for reproducibility.
with open(CONFIG_PATH, "w") as file:
    json.dump(PPE_CONFIG, file, indent=4)

# Verify that the intended values were saved.
with open(CONFIG_PATH, "r") as file:
    saved_config = json.load(file)

print("Epochs:", saved_config["epochs"])
print("Early stopping patience:", saved_config["patience"])
print("Configuration saved:", CONFIG_PATH)
print("Training started: NO")

assert saved_config["epochs"] == 20
assert saved_config["patience"] == 5

print("\nConfiguration verification: PASS")

Epochs: 20
Early stopping patience: 5
Configuration saved: /content/drive/MyDrive/FireWatch-AI/configs/ppe_training_config.json
Training started: NO

Configuration verification: PASS
